<a href="https://colab.research.google.com/github/vitevo/ai-qualitative-research-notebooks/blob/main/comentarios_youtube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descarga de comentarios de YouTube para análisis cualitativo

Cuaderno para reunir los comentarios públicos de uno o varios videos de YouTube y prepararlos como corpus para análisis temático, análisis de contenido o etnografía digital.

**Qué produce**
- `.csv` y `.xlsx` con un comentario por fila: texto, fecha, votos, si es respuesta y a qué comentario responde.
- Autores **seudonimizados** por defecto (`U0001`, `U0002`…), sin nombres de canal ni fotos de perfil.
- Una ficha de recolección (`.txt`) con el video, la fecha y la configuración usada, para reportarla en el método.

**Cómo usarlo.** Ajusta la sección 2 y ejecuta las celdas en orden con ▶. No necesita GPU.

**Antes de usarlo con fines de investigación**, lee la sección *Consideraciones éticas y legales* al final.

## 1. Instalar

In [ ]:
!pip -q install youtube-comment-downloader openpyxl

import itertools, hashlib, datetime, os
import pandas as pd
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR, SORT_BY_RECENT

## 2. Configuración

- **VIDEOS**: uno o varios enlaces. Acepta `youtube.com/watch?v=…`, `youtu.be/…` y `youtube.com/shorts/…`.
- **ORDEN**: `SORT_BY_POPULAR` (los más relevantes primero) o `SORT_BY_RECENT` (los más nuevos primero).
- **MAX_COMENTARIOS**: límite por video, incluidas las respuestas. `None` descarga todos; en videos muy comentados puede tardar bastante.
- **SEUDONIMIZAR**: reemplaza a los autores por códigos. Desactívalo solo si tu diseño y tu comité de ética lo justifican.

In [ ]:
VIDEOS = [
    "https://www.youtube.com/watch?v=E4cLKIxt8W8",
]

ORDEN = SORT_BY_POPULAR
MAX_COMENTARIOS = 2000
INCLUIR_RESPUESTAS = True
SEUDONIMIZAR = True

CARPETA_SALIDA = "comentarios"

## 3. Funciones de apoyo

In [ ]:
import re
from urllib.parse import urlparse, parse_qs

def id_video(url):
    """Extrae el ID de un enlace de YouTube (watch, youtu.be, shorts, embed o live)."""
    u = urlparse(url.strip())
    if u.hostname and u.hostname.endswith("youtu.be"):
        return u.path.lstrip("/").split("/")[0] or None
    if "v" in parse_qs(u.query):
        return parse_qs(u.query)["v"][0]
    m = re.match(r"^/(shorts|embed|live)/([\w-]{11})", u.path or "")
    return m.group(2) if m else None

def a_numero(votos):
    """Convierte '1,2 K', '1.2K', '3 M' o '15' en número entero."""
    t = str(votos).strip().upper().replace(" ", "").replace("MIL", "K")
    if not t:
        return 0
    mult = 1_000 if t.endswith("K") else 1_000_000 if t.endswith("M") else 1
    t = t.rstrip("KM").replace(",", ".")
    try:
        return int(round(float(t) * mult))
    except ValueError:
        return 0

class Seudonimizador:
    """Asigna a cada autor un código estable (U0001, U0002…) dentro de la sesión."""
    def __init__(self):
        self.tabla = {}
    def __call__(self, autor):
        clave = hashlib.sha256(autor.encode("utf-8")).hexdigest()
        if clave not in self.tabla:
            self.tabla[clave] = f"U{len(self.tabla) + 1:04d}"
        return self.tabla[clave]

def a_tabla(comentarios, vid, seudo):
    filas = []
    for c in comentarios:
        cid = c.get("cid", "")
        es_respuesta = bool(c.get("reply")) or "." in cid
        if es_respuesta and not INCLUIR_RESPUESTAS:
            continue
        autor = c.get("author", "")
        filas.append({
            "video_id": vid,
            "comentario_id": cid,
            "responde_a": cid.split(".")[0] if es_respuesta else "",
            "es_respuesta": es_respuesta,
            "autor": seudo(autor) if SEUDONIMIZAR else autor,
            "texto": c.get("text", ""),
            "fecha_relativa": c.get("time", ""),
            "fecha": (datetime.datetime.fromtimestamp(c["time_parsed"], datetime.timezone.utc).date().isoformat()
                      if c.get("time_parsed") else ""),
            "votos": a_numero(c.get("votes", "")),
            "corazon_del_creador": c.get("heart", False),
        })
    return pd.DataFrame(filas)

print("Funciones listas.")

## 4. Descargar

La librería lee los comentarios tal como los muestra la página pública de YouTube, sin usar la API oficial. Si YouTube cambia su página, puede dejar de funcionar; en ese caso actualízala con `!pip install -U youtube-comment-downloader`.

In [ ]:
descargador = YoutubeCommentDownloader()
seudo = Seudonimizador()
tablas = []

for url in VIDEOS:
    vid = id_video(url)
    if not vid:
        print(f"No se reconoce el enlace: {url}")
        continue
    print(f"Descargando comentarios de {vid}...")
    generador = descargador.get_comments_from_url(f"https://www.youtube.com/watch?v={vid}", sort_by=ORDEN)
    if MAX_COMENTARIOS:
        generador = itertools.islice(generador, MAX_COMENTARIOS)
    df_video = a_tabla(generador, vid, seudo)
    print(f"  {len(df_video)} comentarios ({int(df_video['es_respuesta'].sum()) if len(df_video) else 0} respuestas)")
    tablas.append(df_video)

df = pd.concat(tablas, ignore_index=True) if tablas else pd.DataFrame()
df.head(10)

## 5. Revisión rápida

Antes de analizar, conviene mirar el corpus: cuántos comentarios hay, cuántos están vacíos o repetidos (spam, cadenas), y qué largo tienen.

In [ ]:
if len(df):
    df["n_palabras"] = df["texto"].str.split().str.len()
    print("Comentarios:", len(df))
    print("Autores distintos:", df["autor"].nunique())
    print("Vacíos:", int((df["texto"].str.strip() == "").sum()))
    print("Textos duplicados:", int(df["texto"].duplicated().sum()))
    print("Palabras por comentario (mediana):", int(df["n_palabras"].median()))
    print("\nComentarios con más votos:")
    display(df.sort_values("votos", ascending=False)[["autor", "texto", "votos", "fecha"]].head(10))

## 6. Guardar y descargar

El `.xlsx` puede importarse en software de análisis cualitativo que acepte datos de encuesta o documentos tabulares (por ejemplo, Atlas.ti o NVivo), donde cada fila se convierte en un documento o caso.

In [ ]:
from google.colab import files
import shutil

os.makedirs(CARPETA_SALIDA, exist_ok=True)
hoy = datetime.date.today().isoformat()
base = os.path.join(CARPETA_SALIDA, f"comentarios_youtube_{hoy}")

df.to_csv(base + ".csv", index=False, encoding="utf-8-sig")
df.to_excel(base + ".xlsx", index=False)

with open(base + "_ficha.txt", "w", encoding="utf-8") as f:
    f.write(f"Fecha de recolección: {hoy}\n")
    f.write("Videos:\n" + "".join(f"  - {u}\n" for u in VIDEOS))
    f.write(f"Orden: {'populares' if ORDEN == SORT_BY_POPULAR else 'recientes'}\n")
    f.write(f"Límite por video: {MAX_COMENTARIOS}\n")
    f.write(f"Respuestas incluidas: {INCLUIR_RESPUESTAS}\n")
    f.write(f"Autores seudonimizados: {SEUDONIMIZAR}\n")
    f.write(f"Comentarios obtenidos: {len(df)}\n")
    f.write("Herramienta: youtube-comment-downloader (Python), Google Colab\n")

shutil.make_archive("comentarios_youtube", "zip", CARPETA_SALIDA)
files.download("comentarios_youtube.zip")

## Consideraciones éticas y legales

- **Términos de servicio.** Esta librería lee la página pública de YouTube sin pasar por su API oficial, algo que los términos de servicio de YouTube restringen. Para proyectos con financiamiento, publicación de datos o revisión ética exigente, considera la [YouTube Data API v3](https://developers.google.com/youtube/v3), que es la vía autorizada (requiere una clave gratuita y tiene cuotas diarias).
- **Público no significa sin consideraciones.** Aunque los comentarios sean visibles para cualquiera, sus autores no esperan ser citados en una investigación. La seudonimización reduce el riesgo, pero una cita textual puede encontrarse con un buscador y llevar al autor; en la publicación, parafrasea o usa citas breves cuando el tema sea sensible (AoIR, 2020).
- **No publiques el corpus.** No subas los archivos descargados a un repositorio público. Comparte, si hace falta, los ID de los comentarios y el procedimiento, para que otros puedan reconstruirlo.
- **Menores de edad.** YouTube tiene usuarios menores de 18 años y no es posible verificar la edad de quien comenta; tenlo en cuenta en tu protocolo.

franzke, a. s., Bechmann, A., Zimmer, M., Ess, C., & the Association of Internet Researchers. (2020). *Internet research: Ethical guidelines 3.0*. https://aoir.org/reports/ethics3.pdf